# Лабораторная работа №7
## Применение логистической регрессии для задач классификации

Данный ноутбук содержит пошаговое выполнение всех заданий лабораторной работы №7.
Каждый блок кода снабжён подробными комментариями.

## 1. Импорт библиотек

In [1]:
# Импорт библиотеки NumPy для работы с массивами
import numpy as np

# Импорт библиотеки Pandas для работы с таблицами данных
import pandas as pd

# Импорт библиотек для визуализации
import matplotlib.pyplot as plt

# Импорт инструментов scikit-learn для машинного обучения
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

## 2. Загрузка и подготовка данных (Breast Cancer Dataset)

In [2]:
# Импорт встроенного датасета Breast Cancer
from sklearn.datasets import load_breast_cancer

# Загрузка данных в переменные
data = load_breast_cancer()

# Создание DataFrame из признаков
X = pd.DataFrame(data.data, columns=data.feature_names)

# Целевая переменная
y = data.target

# Вывод первых строк датасета
X.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


## 3. Разделение данных на обучающую и тестовую выборки

In [3]:
# Разделяем данные: 70% — обучение, 30% — тест
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

## 4. Создание Pipeline с логистической регрессией

In [4]:
# Создаём конвейер обработки данных
pipeline = Pipeline([
    ('scaler', StandardScaler()),  # Масштабирование признаков
    ('logreg', LogisticRegression(max_iter=1000))  # Логистическая регрессия
])

## 5. Подбор гиперпараметров с помощью GridSearchCV

In [5]:
# Задаём сетку гиперпараметров
param_grid = {
    'logreg__C': [0.01, 0.1, 1, 10],
    'logreg__penalty': ['l2'],
    'logreg__solver': ['lbfgs']
}

# GridSearch с кросс-валидацией
grid = GridSearchCV(
    pipeline,
    param_grid,
    cv=5,
    scoring='f1_weighted'
)

# Обучение модели
grid.fit(X_train, y_train)

GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('logreg',
                                        LogisticRegression(max_iter=1000))]),
             param_grid={'logreg__C': [0.01, 0.1, 1, 10],
                         'logreg__penalty': ['l2'],
                         'logreg__solver': ['lbfgs']},
             scoring='f1_weighted')

## 6. Оценка качества модели

In [6]:
# Получаем лучшие параметры
best_model = grid.best_estimator_

# Предсказание на тестовой выборке
y_pred = best_model.predict(X_test)

# Accuracy модели
accuracy = accuracy_score(y_test, y_pred)

# F1-score модели
f1 = f1_score(y_test, y_pred, average='weighted')

accuracy, f1

(0.9824561403508771, 0.9824844109908538)

## 7. Матрица ошибок и отчёт классификации

In [7]:
# Матрица ошибок
conf_matrix = confusion_matrix(y_test, y_pred)
conf_matrix

# Детальный отчёт классификации
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.97      0.98      0.98        63
           1       0.99      0.98      0.99       108

    accuracy                           0.98       171
   macro avg       0.98      0.98      0.98       171
weighted avg       0.98      0.98      0.98       171



## 8. Выводы
- Логистическая регрессия показала стабильное качество.
- Использование Pipeline предотвращает утечку данных.
- GridSearchCV позволяет подобрать оптимальные параметры модели.